# Music Annotation Project V8.0d
## 33×33 Reproducibility, Full-Pixel OOF and Component-Safety Audit

**Runtime:** GPU. Use **Runtime → Run all**.

V8.0d does not change the image model. It audits the deterministic 33×33 implementation that produced the promising V8.0c control result.

### Registered sequence

1. Reproduce the V8.0b/V8.0c roster, stable labels, centre rows and source-group fold map by hash.
2. Train the unchanged 33×33 CNN twice from fresh model initialisation in this runtime.
3. Compare semantic model hashes, raw OOF scores, calibrated probabilities and decisions exactly.
4. Score every eligible held-out pixel, including truth-side UNCERTAIN rows.
5. Apply the registered no-singleton rule, reconstruct a challenger image, and prove outside-mask and non-PRINT equivalence to V7.8.
6. Analyse authoritative-mask components for catastrophic errors.
7. Calculate page/source-group metrics, Wilson bounds and 10,000-resample page-clustered bootstrap intervals.
8. Produce visual-audit queues and crop sheets.
9. Write a run lock. A later clean-runtime run can compare against the prior completed lock automatically.

If exact same-runtime replay fails, the notebook records failure and may run a predeclared three-seed stability audit. It never lowers thresholds, changes channels, changes labels, uses independent data or permits promotion.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-learn pyarrow torch torchvision matplotlib


## 1. Frozen configuration


In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['PYTHONHASHSEED']='42'

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, random, sys, platform, gc
import cv2, torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
MANIFEST_PATH=ROOT/'development_score_manifest.csv'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_0d'
PREVIOUS_LOCK_PATH=OUTPUT_ROOT/'latest_completed_lock.json'
V75_RUN_OVERRIDE=None

SEED=42
STABILITY_SEEDS=[42,43,44]
RUN_SEED_STABILITY_IF_EXACT_FAIL=True
PATCH=33; HALF=PATCH//2
DARK_THRESHOLD=200; TRUTH_TOLERANCE=1
MAX_PER_PAGE_CLASS=25000
BATCH=512; EPOCHS=12; LR=1e-3; WEIGHT_DECAY=1e-4
PRINT_THRESHOLD=.995; ANNOTATION_THRESHOLD=.995; PRINT_MARGIN=.990
BOOTSTRAP_RESAMPLES=10000
CROP_RADIUS=55
EXPECTED_MANIFEST_SHA256='8ad9490658edcf898da7b04b2820d858a12bf4ec2ce4ab3a722d762a2831bcd7'
EXPECTED_FOLD_MAP_SHA256='ae1ac4557c1ffab1371094f3fe1c818fd0f652bb271512830ee06da55fc4867b'
CHANNELS=['obs','norm','mask','depth','horizontal','vertical','compact','preserve','remove','ambiguous','edge']

DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE!='cuda': raise RuntimeError('Select a GPU runtime.')
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_0d_33px_replay_audit'
OUT=OUTPUT_ROOT/RUN
TABLES=OUT/'tables'; MODELS=OUT/'models'; AUDIT=OUT/'audit'; CROPS=OUT/'visual_audit'
for p in [OUT,TABLES,MODELS,AUDIT,CROPS]: p.mkdir(parents=True,exist_ok=False)
print('Device:',torch.cuda.get_device_name(0)); print('Output:',OUT)


## 2. Integrity helpers and authoritative input gate


In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def latest(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def image_map(folder):
    folder=Path(folder)
    return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if folder.exists() else {}

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def object_hash(x): return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a)
    h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def semantic_model_hash(model):
    h=hashlib.sha256()
    for key,tensor in sorted(model.state_dict().items()):
        a=np.ascontiguousarray(tensor.detach().cpu().numpy())
        h.update(key.encode()); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes())
    return h.hexdigest()
def wilson_upper(errors,n,z=1.96):
    if n==0:return np.nan
    p=errors/n; d=1+z*z/n; c=p+z*z/(2*n); r=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n)); return (c+r)/d

def reset_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

V75=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else latest(V75_ROOT,['tables/cache_status.csv'])
if V75 is None: raise RuntimeError('Complete V7.5 run not found.')
if not MANIFEST_PATH.exists(): raise RuntimeError(f'Missing {MANIFEST_PATH}')
if sha256_file(MANIFEST_PATH)!=EXPECTED_MANIFEST_SHA256: raise RuntimeError('Development manifest hash differs from V8.0b/V8.0c.')
manifest=pd.read_csv(MANIFEST_PATH,dtype=str)
required={'page','source_group','score_title','edition_or_source','acquisition_notes'}
if required-set(manifest): raise RuntimeError(f'Manifest missing {sorted(required-set(manifest))}')
if manifest.page.duplicated().any() or manifest.source_group.isna().any() or (manifest.source_group.str.strip()=='').any():
    raise RuntimeError('Duplicate pages or incomplete source groups.')
status=pd.read_csv(V75/'tables'/'cache_status.csv'); cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}
clean_map=image_map(ROOT/'source'/'clean'); observed_map=image_map(ROOT/'aligned')
pages=sorted(set(cache)&set(clean_map)&set(observed_map))
if set(manifest.page)!=set(pages): raise RuntimeError('Manifest and frozen V7.5 roster differ.')
meta=manifest.set_index('page')
print('Pages:',len(pages),'source groups:',manifest.source_group.nunique())


## 3. Reproduce labels and observable channels


In [ ]:
def printed_ink(clean): return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def truth_maps(clean):
    ink=printed_ink(clean); k=np.ones((3,3),np.uint8)
    near=cv2.dilate(ink.astype(np.uint8),k)>0; eroded=cv2.erode(ink.astype(np.uint8),k)>0
    return ink,near,np.logical_xor(near,eroded)
def background_surface(obs,mask,sigma=45.):
    vis=(~mask)&(obs>=200); num=cv2.GaussianBlur(obs.astype(np.float32)*vis,(0,0),sigma)
    den=cv2.GaussianBlur(vis.astype(np.float32),(0,0),sigma); fallback=cv2.GaussianBlur(obs,(0,0),sigma)
    return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)

runtime={}; label_audit=[]
req=['mask','geometry_t80','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']
for page in pages:
    with np.load(cache[page]) as z:
        miss=[k for k in req if k not in z.files]
        if miss: raise RuntimeError(f'{page}: missing {miss}')
        a={k:z[k] for k in req}
    clean=cv2.imread(str(clean_map[page]),0); obs=cv2.imread(str(observed_map[page]),0)
    if clean is None or obs is None or clean.shape!=obs.shape or any(v.shape!=obs.shape for v in a.values()): raise RuntimeError(f'{page}: read/shape failure')
    mask=a['mask']>0
    if not np.array_equal(a['geometry_t80'][~mask],obs[~mask]): raise RuntimeError(f'{page}: outside-mask baseline drift')
    ink,near,boundary=truth_maps(clean); eligible=mask&(obs<DARK_THRESHOLD)&~(a['g80_preserve']>0)
    label=np.full(obs.shape,-1,np.int8); label[eligible&near&~boundary]=1; label[eligible&~near&~boundary]=0; label[eligible&boundary]=2
    paper=background_surface(obs,mask); norm=cv2.divide(obs,np.maximum(paper,1),scale=255)
    runtime[page]={'obs':obs,'clean':clean,'mask':mask,'baseline':a['geometry_t80'],'label':label,
        'norm':norm,'depth':cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5),'edge':cv2.Canny(obs,70,170)>0,
        'preserve':a['g80_preserve']>0,'remove':a['g80_remove']>0,'ambiguous':a['g80_ambiguous']>0,
        'horizontal':a['g80_horizontal']>0,'vertical':a['g80_vertical']>0,'compact':a['g80_compact']>0}
    label_audit.append({'page':page,'source_group':meta.loc[page,'source_group'],'eligible':int(eligible.sum()),
        'print':int((label==1).sum()),'annotation':int((label==0).sum()),'uncertain_truth':int((label==2).sum()),'cache_sha256':sha256_file(cache[page])})
pd.DataFrame(label_audit).to_csv(AUDIT/'label_and_cache_audit.csv',index=False)
print(pd.DataFrame(label_audit)[['eligible','print','annotation','uncertain_truth']].sum())


## 4. Reproduce the stable training roster, all eligible roster and source-group folds


In [ ]:
rng=np.random.default_rng(SEED); stable=[]; all_rows=[]
for page,z in runtime.items():
    for cls in [0,1,2]:
        ys,xs=np.where(z['label']==cls)
        for y,x in zip(ys,xs): all_rows.append({'page':page,'source_group':str(meta.loc[page,'source_group']),'y':int(y),'x':int(x),'truth':cls})
    for cls in [0,1]:
        ys,xs=np.where(z['label']==cls); idx=np.arange(len(ys)); take=idx if len(idx)<=MAX_PER_PAGE_CLASS else rng.choice(idx,MAX_PER_PAGE_CLASS,replace=False)
        for j in take: stable.append({'page':page,'source_group':str(meta.loc[page,'source_group']),'y':int(ys[j]),'x':int(xs[j]),'truth':cls})
stable=pd.DataFrame(stable).sort_values(['source_group','page','y','x']).reset_index(drop=True); stable['sample_id']=np.arange(len(stable))
all_candidates=pd.DataFrame(all_rows).sort_values(['source_group','page','y','x']).reset_index(drop=True); all_candidates['candidate_id']=np.arange(len(all_candidates))
groups=stable.source_group; splitter=LeaveOneGroupOut() if groups.nunique()<=10 else GroupKFold(n_splits=min(10,groups.nunique()))
fold=np.full(len(stable),-1,int); split_rows=[]; group_to_fold={}
for f,(tr,te) in enumerate(splitter.split(stable,stable.truth,groups),1):
    train_g=set(groups.iloc[tr]); test_g=set(groups.iloc[te]); overlap=train_g&test_g
    if overlap: raise RuntimeError(f'Fold {f} leakage {overlap}')
    fold[te]=f
    for g in test_g:
        if g in group_to_fold: raise RuntimeError('Source group assigned twice.')
        group_to_fold[g]=f
    split_rows.append({'fold':f,'train_groups':len(train_g),'test_groups':len(test_g),'overlap_groups':len(overlap),'test_rows':len(te),'test_pages':stable.iloc[te].page.nunique()})
stable['fold']=fold; all_candidates['fold']=all_candidates.source_group.map(group_to_fold)
if all_candidates.fold.isna().any(): raise RuntimeError('Eligible candidate belongs to a group absent from stable supervised rows.')
all_candidates.fold=all_candidates.fold.astype(int)
fold_payload=stable[['sample_id','page','source_group','fold']].to_dict('records'); fold_hash=object_hash(fold_payload)
if fold_hash!=EXPECTED_FOLD_MAP_SHA256: raise RuntimeError(f'Fold hash mismatch {fold_hash}')
stable.to_csv(TABLES/'stable_sample_manifest.csv',index=False); all_candidates.to_csv(TABLES/'all_eligible_manifest.csv',index=False)
pd.DataFrame(split_rows).to_csv(AUDIT/'source_group_split_audit.csv',index=False)
print('Stable:',len(stable),'all eligible:',len(all_candidates),'fold hash:',fold_hash)


## 5. Shared 33×33 input implementation


In [ ]:
padded={}
for page,z in runtime.items():
    padded[page]={}
    for key in CHANNELS:
        a=z[key].astype(np.float32)
        if key in ['obs','norm']: a/=255.0
        padded[page][key]=np.pad(a,HALF,mode='reflect')

page_list=pages; page_index={p:i for i,p in enumerate(page_list)}
def encode(df):
    return df.page.map(page_index).to_numpy(np.int16),df.y.to_numpy(np.int32),df.x.to_numpy(np.int32),df.truth.to_numpy(np.int8)
stable_pid,stable_y,stable_x,stable_truth=encode(stable)
all_pid,all_y,all_x,all_truth=encode(all_candidates)

def get_patch(pid,y,x):
    page=page_list[int(pid)]
    return np.stack([padded[page][k][y:y+PATCH,x:x+PATCH] for k in CHANNELS]).astype(np.float32)
for j in np.linspace(0,len(stable)-1,100,dtype=int):
    x=get_patch(stable_pid[j],stable_y[j],stable_x[j]); assert x.shape==(11,33,33)
    assert np.isclose(x[0,16,16],runtime[stable.iloc[j].page]['obs'][stable_y[j],stable_x[j]]/255.)
print('Patch identity checks passed.')


## 6. Fixed CNN, deterministic loader and semantic hashing


In [ ]:
class ArrayPatchDataset(Dataset):
    def __init__(self,indices,pids,ys,xs,truth): self.idx=np.asarray(indices); self.pids=pids; self.ys=ys; self.xs=xs; self.truth=truth
    def __len__(self): return len(self.idx)
    def __getitem__(self,i):
        j=int(self.idx[i]); return torch.from_numpy(get_patch(self.pids[j],self.ys[j],self.xs[j])),torch.tensor(int(self.truth[j]),dtype=torch.long)
class TopologyCNN(nn.Module):
    def __init__(self,nc=11):
        super().__init__(); self.net=nn.Sequential(
            nn.Conv2d(nc,24,3,padding=1),nn.ReLU(),nn.BatchNorm2d(24),nn.MaxPool2d(2),
            nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.BatchNorm2d(48),nn.MaxPool2d(2),
            nn.Conv2d(48,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1)); self.head=nn.Linear(64,2)
    def forward(self,x): return self.head(self.net(x).flatten(1))
def worker_init(worker_id):
    s=SEED+worker_id; random.seed(s); np.random.seed(s)
def loader(indices,pids,ys,xs,truth,shuffle,seed):
    g=torch.Generator(); g.manual_seed(seed)
    return DataLoader(ArrayPatchDataset(indices,pids,ys,xs,truth),batch_size=BATCH,shuffle=shuffle,
        num_workers=2,pin_memory=True,generator=g,worker_init_fn=worker_init,persistent_workers=False)
def fit_fold(train_idx,seed):
    reset_seed(seed); model=TopologyCNN().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    counts=np.bincount(stable_truth[train_idx],minlength=2); weights=torch.tensor(counts.sum()/np.maximum(counts,1),dtype=torch.float32,device=DEVICE)
    loss_fn=nn.CrossEntropyLoss(weight=weights); hist=[]
    for ep in range(EPOCHS):
        total=0.; model.train()
        for xb,yb in loader(train_idx,stable_pid,stable_y,stable_x,stable_truth,True,seed):
            xb=xb.to(DEVICE,non_blocking=True); yb=yb.to(DEVICE,non_blocking=True); opt.zero_grad(set_to_none=True)
            loss=loss_fn(model(xb),yb); loss.backward(); opt.step(); total+=loss.detach().item()*len(yb)
        hist.append(total/len(train_idx))
    return model,hist
def predict(model,indices,pids,ys,xs,truth):
    out=[]; model.eval()
    with torch.no_grad():
        for xb,_ in loader(indices,pids,ys,xs,truth,False,SEED): out.extend(torch.softmax(model(xb.to(DEVICE,non_blocking=True)),1)[:,1].cpu().numpy())
    return np.asarray(out,dtype=np.float64)


## 7. Two fresh same-seed OOF replicas and full eligible scoring


In [ ]:
def run_replica(replica,seed=SEED):
    raw_stable=np.full(len(stable),np.nan); raw_all=np.full(len(all_candidates),np.nan); model_rows=[]; histories=[]
    for f in sorted(stable.fold.unique()):
        tr=np.where(stable.fold.to_numpy()!=f)[0]; te=np.where(stable.fold.to_numpy()==f)[0]; ae=np.where(all_candidates.fold.to_numpy()==f)[0]
        model,hist=fit_fold(tr,seed); sem=semantic_model_hash(model)
        raw_stable[te]=predict(model,te,stable_pid,stable_y,stable_x,stable_truth)
        raw_all[ae]=predict(model,ae,all_pid,all_y,all_x,all_truth)
        path=MODELS/f'replica{replica}_fold{f}_seed{seed}.pt'; torch.save(model.state_dict(),path)
        model_rows.append({'replica':replica,'seed':seed,'fold':int(f),'semantic_model_sha256':sem,'checkpoint_sha256':sha256_file(path),'train_rows':len(tr),'test_rows':len(te),'eligible_test_rows':len(ae)})
        for ep,v in enumerate(hist,1): histories.append({'replica':replica,'seed':seed,'fold':int(f),'epoch':ep,'mean_loss':v})
        del model; gc.collect(); torch.cuda.empty_cache()
    if not np.isfinite(raw_stable).all() or not np.isfinite(raw_all).all(): raise RuntimeError('Incomplete OOF predictions.')
    return raw_stable,raw_all,pd.DataFrame(model_rows),pd.DataFrame(histories)

raw_s1,raw_a1,models1,hist1=run_replica(1,SEED)
raw_s2,raw_a2,models2,hist2=run_replica(2,SEED)
pd.concat([models1,models2]).to_csv(AUDIT/'model_hashes.csv',index=False)
pd.concat([hist1,hist2]).to_csv(TABLES/'training_history.csv',index=False)
print('Training replicas complete.')


## 8. Cross-fitted calibration, exact replay and external prior-lock comparison


In [ ]:
def calibrate(raw_stable,raw_all):
    ps=np.full(len(stable),np.nan); pa=np.full(len(all_candidates),np.nan); calibrator_hashes=[]
    for f in sorted(stable.fold.unique()):
        tr=np.where(stable.fold.to_numpy()!=f)[0]; te=np.where(stable.fold.to_numpy()==f)[0]; ae=np.where(all_candidates.fold.to_numpy()==f)[0]
        iso=IsotonicRegression(out_of_bounds='clip').fit(raw_stable[tr],stable_truth[tr])
        ps[te]=iso.predict(raw_stable[te]); pa[ae]=iso.predict(raw_all[ae])
        payload={'x':iso.X_thresholds_.tolist(),'y':iso.y_thresholds_.tolist()}; calibrator_hashes.append({'fold':int(f),'sha256':object_hash(payload)})
    return ps,pa,calibrator_hashes
def decide(p):
    p=np.asarray(p); q=1-p; out=np.full(len(p),'UNCERTAIN',object); out[q>=ANNOTATION_THRESHOLD]='ANNOTATION'; out[(p>=PRINT_THRESHOLD)&((p-q)>=PRINT_MARGIN)]='PRINT'; return out
p_s1,p_a1,cal1=calibrate(raw_s1,raw_a1); p_s2,p_a2,cal2=calibrate(raw_s2,raw_a2)
d_s1,d_a1=decide(p_s1),decide(p_a1); d_s2,d_a2=decide(p_s2),decide(p_a2)
checks={
 'stable_raw_array_equal':bool(np.array_equal(raw_s1,raw_s2)), 'all_raw_array_equal':bool(np.array_equal(raw_a1,raw_a2)),
 'stable_probability_equal':bool(np.array_equal(p_s1,p_s2)), 'all_probability_equal':bool(np.array_equal(p_a1,p_a2)),
 'stable_decision_equal':bool(np.array_equal(d_s1,d_s2)), 'all_decision_equal':bool(np.array_equal(d_a1,d_a2)),
 'semantic_model_hashes_equal':bool(models1.semantic_model_sha256.tolist()==models2.semantic_model_sha256.tolist()),
 'calibrator_hashes_equal':bool(cal1==cal2)}
checks['same_runtime_exact_replay_pass']=all(checks.values())
replica_lock={'fold_map_sha256':fold_hash,'manifest_sha256':sha256_file(MANIFEST_PATH),'stable_raw_sha256':array_hash(raw_s1),
 'all_raw_sha256':array_hash(raw_a1),'stable_probability_sha256':array_hash(p_s1),'all_probability_sha256':array_hash(p_a1),
 'stable_decision_sha256':object_hash(d_s1.tolist()),'all_decision_sha256':object_hash(d_a1.tolist()),
 'semantic_model_hashes':models1.semantic_model_sha256.tolist(),'calibrator_hashes':cal1}
run_lock_sha=object_hash(replica_lock); previous=json.loads(PREVIOUS_LOCK_PATH.read_text()) if PREVIOUS_LOCK_PATH.exists() else None
external_match=bool(previous and previous.get('run_lock_sha256')==run_lock_sha)
replay={'checks':checks,'run_lock_sha256':run_lock_sha,'previous_completed_lock_found':previous is not None,'previous_clean_run_match':external_match}
(AUDIT/'replay_test.json').write_text(json.dumps(replay,indent=2)); print(json.dumps(replay,indent=2))


## 9. Optional seed-stability audit only if exact same-runtime replay fails


In [ ]:
stability_rows=[]
if not checks['same_runtime_exact_replay_pass'] and RUN_SEED_STABILITY_IF_EXACT_FAIL:
    seed_results={SEED:(raw_s1,p_s1,d_s1)}
    for seed in STABILITY_SEEDS[1:]:
        rs,ra,_,_=run_replica(f'seed{seed}',seed); ps,pa,_=calibrate(rs,ra); seed_results[seed]=(rs,ps,decide(ps))
    for seed,(rs,ps,ds) in seed_results.items():
        sel=ds=='PRINT'; stability_rows.append({'seed':seed,'raw_sha256':array_hash(rs),'probability_sha256':array_hash(ps),
            'decision_sha256':object_hash(ds.tolist()),'selected':int(sel.sum()),'true_print':int(np.sum(sel&(stable_truth==1))),
            'false_annotation':int(np.sum(sel&(stable_truth==0)))})
    base=seed_results[SEED][2]=='PRINT'
    for row in stability_rows:
        other=seed_results[row['seed']][2]=='PRINT'; row['print_jaccard_vs_seed42']=float(np.sum(base&other)/max(np.sum(base|other),1))
pd.DataFrame(stability_rows).to_csv(TABLES/'seed_stability_summary.csv',index=False)
print('Seed audit rows:',len(stability_rows))


## 10. Operational no-singleton rule and all-eligible OOF table


In [ ]:
# Use replica 1 only after exact same-runtime replay; otherwise outputs remain diagnostic and freeze stays prohibited.
full=all_candidates.copy(); full['raw_print']=raw_a1; full['p_print']=p_a1; full['decision_pre_run_rule']=d_a1
full['decision']=full.decision_pre_run_rule.copy(); singleton_demotions=0
for page,g in full.groupby('page'):
    z=runtime[page]; pm=np.zeros(z['obs'].shape,np.uint8)
    idx=g.index[g.decision_pre_run_rule=='PRINT']; pm[full.loc[idx,'y'],full.loc[idx,'x']]=1
    n,lab,stats,_=cv2.connectedComponentsWithStats(pm,8)
    for cid in range(1,n):
        if stats[cid,cv2.CC_STAT_AREA]<2:
            ys,xs=np.where(lab==cid); hit=g.index[(g.y.isin(ys))]  # narrowed below by coordinate exactness
            coord={(int(y),int(x)) for y,x in zip(ys,xs)}
            exact=[i for i in idx if (int(full.at[i,'y']),int(full.at[i,'x'])) in coord]
            full.loc[exact,'decision']='UNCERTAIN'; singleton_demotions+=len(exact)
full.to_parquet(TABLES/'all_eligible_oof_predictions.parquet',index=False)
full.to_csv(TABLES/'all_eligible_oof_predictions.csv',index=False)
print(full.decision.value_counts(),'singleton demotions',singleton_demotions)


## 11. V7.8 equivalence and immutable challenger reconstruction


In [ ]:
immutability=[]
for page,g in full.groupby('page'):
    z=runtime[page]; out=z['baseline'].copy(); sel=g[g.decision=='PRINT']; out[sel.y,sel.x]=z['obs'][sel.y,sel.x]
    print_mask=np.zeros(z['mask'].shape,bool); print_mask[sel.y,sel.x]=True
    outside=int(np.sum(out[~z['mask']]!=z['obs'][~z['mask']]))
    nonprint=int(np.sum(out[~print_mask]!=z['baseline'][~print_mask]))
    if outside or nonprint: raise RuntimeError(f'{page}: immutability failure outside={outside} nonprint={nonprint}')
    cv2.imwrite(str(AUDIT/f'{page}_challenger.png'),out)
    immutability.append({'page':page,'outside_mask_drift':outside,'non_print_v78_differences':nonprint,'print_pixels':len(sel)})
pd.DataFrame(immutability).to_csv(AUDIT/'immutability_audit.csv',index=False)


## 12. Authoritative-mask component analysis and catastrophic-error audit


In [ ]:
component_rows=[]
for page,g in full.groupby('page'):
    z=runtime[page]; n,labels,stats,_=cv2.connectedComponentsWithStats(z['mask'].astype(np.uint8),8)
    sel=g[g.decision=='PRINT']
    for cid in sorted(set(labels[sel.y,sel.x].tolist())-{0}):
        q=sel[labels[sel.y,sel.x]==cid]; tp=int((q.truth==1).sum()); fp=int((q.truth==0).sum()); uncertain=int((q.truth==2).sum())
        catastrophic=bool(fp>=50 or (fp/max(tp,1)>1.0 and fp>=20))
        component_rows.append({'page':page,'source_group':meta.loc[page,'source_group'],'component_id':int(cid),
            'mask_area':int(stats[cid,cv2.CC_STAT_AREA]),'selected':len(q),'true_print':tp,'false_annotation':fp,
            'truth_uncertain':uncertain,'false_true_ratio':fp/max(tp,1),'utility':tp-5*fp-(500 if catastrophic else 0),
            'catastrophic':catastrophic,'x':int(stats[cid,cv2.CC_STAT_LEFT]),'y':int(stats[cid,cv2.CC_STAT_TOP]),
            'w':int(stats[cid,cv2.CC_STAT_WIDTH]),'h':int(stats[cid,cv2.CC_STAT_HEIGHT])})
components=pd.DataFrame(component_rows)
components.to_csv(TABLES/'selected_mask_components.csv',index=False)
components[components.catastrophic].to_csv(TABLES/'catastrophic_components.csv',index=False)
print('Selected components:',len(components),'catastrophic:',int(components.catastrophic.sum()) if len(components) else 0)


## 13. Pooled, page, source-group and page-clustered bootstrap metrics


In [ ]:
stable_scored=full[full.truth.isin([0,1])].copy(); selected=stable_scored.decision=='PRINT'
tp=int(np.sum(selected&(stable_scored.truth==1))); fp=int(np.sum(selected&(stable_scored.truth==0))); n=int(selected.sum())
cat=int(components.catastrophic.sum()) if len(components) else 0
summary={'selected':n,'true_print':tp,'false_annotation':fp,'false_true_ratio':fp/max(tp,1),
 'false_preservation_rate':fp/max(n,1),'wilson95_upper':wilson_upper(fp,n),'utility':tp-5*fp-500*cat,
 'pages':int(stable_scored.loc[selected,'page'].nunique()),'source_groups':int(stable_scored.loc[selected,'source_group'].nunique()),
 'eligible_print_coverage':n/len(stable_scored),'uncertain_rate':float((full.decision=='UNCERTAIN').mean()),'catastrophic_components':cat}
page_rows=[]
for page,g in stable_scored.groupby('page'):
    s=g.decision=='PRINT'; t=int(np.sum(s&(g.truth==1))); f=int(np.sum(s&(g.truth==0)))
    page_rows.append({'page':page,'source_group':g.source_group.iloc[0],'eligible':len(g),'selected':int(s.sum()),'true_print':t,'false_annotation':f,'utility':t-5*f})
page_df=pd.DataFrame(page_rows); page_df.to_csv(TABLES/'results_by_page.csv',index=False)
group_df=page_df.groupby('source_group',as_index=False).agg(pages=('page','nunique'),eligible=('eligible','sum'),selected=('selected','sum'),true_print=('true_print','sum'),false_annotation=('false_annotation','sum'),utility=('utility','sum'))
group_df['false_true_ratio']=group_df.false_annotation/np.maximum(group_df.true_print,1); group_df.to_csv(TABLES/'results_by_source_group.csv',index=False)

rng=np.random.default_rng(SEED); unique_pages=page_df.page.to_numpy(); boot=[]
for _ in range(BOOTSTRAP_RESAMPLES):
    sampled=rng.choice(unique_pages,size=len(unique_pages),replace=True); q=page_df.set_index('page').loc[sampled]
    bt=int(q.true_print.sum()); bf=int(q.false_annotation.sum()); bn=bt+bf
    boot.append((bt-5*bf,bf/max(bt,1),bf/max(bn,1)))
b=np.asarray(boot); summary['bootstrap_utility_95_low']=float(np.quantile(b[:,0],.025)); summary['bootstrap_utility_95_high']=float(np.quantile(b[:,0],.975))
summary['bootstrap_false_true_95_high']=float(np.quantile(b[:,1],.975)); summary['bootstrap_false_preservation_95_high']=float(np.quantile(b[:,2],.975))
(OUT/'development_summary.json').write_text(json.dumps(summary,indent=2)); print(json.dumps(summary,indent=2))


## 14. Visual-audit queues and crop sheets


In [ ]:
# Component queues plus uncertain pixels nearest the PRINT threshold.
if len(components):
    components.sort_values(['false_annotation','selected'],ascending=False).head(50).to_csv(TABLES/'visual_audit_false_heavy_components.csv',index=False)
    components.sort_values(['true_print','utility'],ascending=False).head(50).to_csv(TABLES/'visual_audit_beneficial_components.csv',index=False)
uncertain=full[full.decision=='UNCERTAIN'].copy(); uncertain['distance_to_print_threshold']=(PRINT_THRESHOLD-uncertain.p_print).abs()
uncertain.sort_values('distance_to_print_threshold').head(50).to_csv(TABLES/'visual_audit_uncertain_pixels.csv',index=False)

def render_queue(df,name,max_items=50):
    if df is None or len(df)==0:return
    rows=[]
    for _,r in df.head(max_items).iterrows():
        page=r.page; cy=int(r.y+r.h/2) if 'h' in r and pd.notna(r.h) else int(r.y); cx=int(r.x+r.w/2) if 'w' in r and pd.notna(r.w) else int(r.x)
        z=runtime[page]; y0=max(0,cy-CROP_RADIUS); y1=min(z['obs'].shape[0],cy+CROP_RADIUS+1); x0=max(0,cx-CROP_RADIUS); x1=min(z['obs'].shape[1],cx+CROP_RADIUS+1)
        obs=cv2.cvtColor(z['obs'][y0:y1,x0:x1],cv2.COLOR_GRAY2RGB); base=cv2.cvtColor(z['baseline'][y0:y1,x0:x1],cv2.COLOR_GRAY2RGB)
        rows.append((page,cy,cx,obs,base))
    cols=4; fig,axes=plt.subplots(math.ceil(len(rows)/cols)*2,cols,figsize=(16,4*math.ceil(len(rows)/cols)*2),squeeze=False)
    for ax in axes.flat:ax.axis('off')
    k=0
    for page,cy,cx,obs,base in rows:
        for image,title in [(obs,f'{page} observed ({cy},{cx})'),(base,'V7.8 baseline')]: axes.flat[k].imshow(image); axes.flat[k].set_title(title,fontsize=8); axes.flat[k].axis('off'); k+=1
    plt.tight_layout(); plt.savefig(CROPS/f'{name}.png',dpi=150,bbox_inches='tight'); plt.close(fig)
if len(components):
    render_queue(components.sort_values(['false_annotation','selected'],ascending=False),'false_heavy_components')
    render_queue(components.sort_values(['true_print','utility'],ascending=False),'beneficial_components')
render_queue(uncertain.sort_values('distance_to_print_threshold'),'uncertain_pixels')
print('Audit queues written.')


## 15. Development-gate report and run lock

V8.0d cannot satisfy the five-member disagreement or pre-registered OOD requirements because those mechanisms were not implemented in V8.0b/V8.0c. Those are reported as unmet, not silently treated as passes.


In [ ]:
# Gate assessment. Sparse/hard-negative status is not available in the manifest, so D9 is not assessed as passed.
gates={
 'D1_integrity':bool(checks['same_runtime_exact_replay_pass'] and np.isfinite(raw_a1).all()),
 'D2_immutability':bool(pd.DataFrame(immutability)[['outside_mask_drift','non_print_v78_differences']].to_numpy().sum()==0),
 'D3_catastrophic_safety':bool(cat==0),
 'D4_false_true':bool(summary['false_true_ratio']<=.10 and summary['bootstrap_false_true_95_high']<=.20),
 'D5_pixel_safety':bool(summary['false_preservation_rate']<=.005 and summary['wilson95_upper']<=.01),
 'D6_positive_utility':bool(summary['utility']>0 and summary['bootstrap_utility_95_low']>0),
 'D7_nontrivial_benefit':bool(tp>=500 and summary['pages']>=10 and summary['source_groups']>=3),
 'D8_source_stability':bool(((group_df.false_true_ratio<=.25)| (group_df.pages<3)).all() and cat==0),
 'D9_sparse_hard_negative':'NOT_ASSESSED_NO_PREDECLARED_STATUS',
 'D10_uncertainty_reported':True,
 'D11_baseline_dominance':'NOT_ASSESSED_IN_THIS_REPLAY_NOTEBOOK',
 'D12_visual_audit':'QUEUES_GENERATED_REVIEW_NOT_COMPLETED',
 'five_member_disagreement_rule':'NOT_IMPLEMENTED',
 'ood_rule':'NOT_IMPLEMENTED'}
numeric_pass=all(v is True for v in gates.values() if isinstance(v,bool))
freeze_permitted=False
report={'gates':gates,'all_boolean_gates_pass':numeric_pass,'previous_clean_run_match':external_match,
 'freeze_permitted':freeze_permitted,'reason':'Research audit only; full specification is not satisfied and human visual review is outstanding.'}
(OUT/'development_gate_report.json').write_text(json.dumps(report,indent=2))
manifest_out={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.0d-33px-reproducibility-full-pixel-audit',
 'development_only':True,'pages':len(pages),'declared_source_groups':int(manifest.source_group.nunique()),
 'stable_rows':len(stable),'all_eligible_rows':len(all_candidates),'patch':PATCH,'channels':CHANNELS,
 'manifest_sha256':sha256_file(MANIFEST_PATH),'fold_map_sha256':fold_hash,'run_lock_sha256':run_lock_sha,
 'same_runtime_exact_replay_pass':checks['same_runtime_exact_replay_pass'],'previous_clean_run_match':external_match,
 'full_pixel_scoring_completed':True,'component_audit_completed':True,'independent_data_used':False,
 'freeze_permitted':False,'software':{'python':sys.version,'torch':torch.__version__,'opencv':cv2.__version__,
 'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform(),'gpu':torch.cuda.get_device_name(0)}}
(OUT/'experiment_manifest.json').write_text(json.dumps(manifest_out,indent=2))
completed={'run_lock_sha256':run_lock_sha,'run_dir':str(OUT),'created_utc':manifest_out['created_utc'],'manifest_sha256':manifest_out['manifest_sha256'],'fold_map_sha256':fold_hash}
tmp=Path(str(PREVIOUS_LOCK_PATH)+'.tmp'); tmp.write_text(json.dumps(completed,indent=2)); tmp.replace(PREVIOUS_LOCK_PATH)
print(json.dumps(report,indent=2))


## Outputs to share

Please share:

- `experiment_manifest.json`
- `development_summary.json`
- `development_gate_report.json`
- `audit/replay_test.json`
- `audit/model_hashes.csv`
- `audit/source_group_split_audit.csv`
- `audit/immutability_audit.csv`
- `tables/selected_mask_components.csv`
- `tables/catastrophic_components.csv`
- `tables/results_by_page.csv`
- `tables/results_by_source_group.csv`
- `tables/visual_audit_false_heavy_components.csv`
- `tables/visual_audit_beneficial_components.csv`
- `tables/visual_audit_uncertain_pixels.csv`
- the three PNG files in `visual_audit/`

### Clean-runtime replay

Run this notebook a second time in a fresh Colab runtime without changing any constants. The second run will find `latest_completed_lock.json` from the first run and report `previous_clean_run_match` in `audit/replay_test.json`. This is separate from the two fresh same-seed model replicas already trained inside each run.
